# Real-Time Performance Evaluation: eBPF + ML Bottleneck Diagnosis

This notebook analyzes the CSV files produced by the benchmark runner:

- `runtime_benchmark_results/cycle_metrics.csv` — per-cycle timings and monitoring intervals
- `runtime_benchmark_results/run_summary.csv` — optional run-level summaries
- `runtime_benchmark_results/system_resource_metrics.csv` — optional CPU/memory measurements

It is designed to work with the column names proposed in the GitHub Copilot benchmark prompt and common variants. If a column is not detected, inspect the column names printed below and update the aliases in the configuration cell.

**Important:** The notebook analyzes measurements; it does not create or simulate benchmark results. Missing data is reported as unavailable. A single run can characterize that run, but cannot establish repeatability or broad generalization.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Change this if your benchmark output is stored elsewhere.
RESULTS_DIR = Path("runtime_benchmark_results")

CYCLE_FILE = RESULTS_DIR / "cycle_metrics.csv"
SUMMARY_FILE = RESULTS_DIR / "run_summary.csv"
RESOURCE_FILE = RESULTS_DIR / "system_resource_metrics.csv"

# If running the notebook from inside the results folder, allow that layout too.
if not CYCLE_FILE.exists() and Path("cycle_metrics.csv").exists():
    RESULTS_DIR = Path(".")
    CYCLE_FILE = RESULTS_DIR / "cycle_metrics.csv"
    SUMMARY_FILE = RESULTS_DIR / "run_summary.csv"
    RESOURCE_FILE = RESULTS_DIR / "system_resource_metrics.csv"

if not CYCLE_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {CYCLE_FILE}. Put cycle_metrics.csv in "
        f"{RESULTS_DIR.resolve()} or update RESULTS_DIR above."
    )

cycles = pd.read_csv(CYCLE_FILE)
summary = pd.read_csv(SUMMARY_FILE) if SUMMARY_FILE.exists() else None
resources = pd.read_csv(RESOURCE_FILE) if RESOURCE_FILE.exists() else None

print("Cycle metrics:", cycles.shape)
print("Cycle columns:", cycles.columns.tolist())
print("\nRun summary:", None if summary is None else summary.shape)
print("Resource metrics:", None if resources is None else resources.shape)


## 1. Normalize common column-name variants

The following cell looks for common aliases for total diagnosis latency, actual monitoring interval, configured interval, timestamps, deadline misses, and stage timings. It converts timing columns ending in `_ns` to milliseconds. Review the detected fields before relying on the plots.


In [ ]:
def norm_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_")

original_columns = list(cycles.columns)
col_lookup = {norm_name(c): c for c in original_columns}

ALIASES = {
    "total_latency": [
        "total_cycle_ms", "total_latency_ms", "total_diagnosis_ms",
        "diagnosis_latency_ms", "total_ms", "cycle_duration_ms",
        "total_cycle_ns", "total_latency_ns", "total_diagnosis_ns",
        "diagnosis_latency_ns", "total_ns"
    ],
    "actual_interval": [
        "actual_cycle_interval_ms", "actual_interval_ms", "cycle_interval_ms",
        "observed_interval_ms", "start_to_start_ms", "actual_cycle_interval_ns",
        "actual_interval_ns", "cycle_interval_ns", "observed_interval_ns"
    ],
    "configured_interval": [
        "configured_interval_ms", "monitoring_interval_ms",
        "target_interval_ms", "configured_interval_ns",
        "monitoring_interval_ns", "target_interval_ns"
    ],
    "timestamp": [
        "cycle_start_time", "cycle_start_timestamp", "timestamp",
        "start_timestamp", "cycle_start", "start_time", "elapsed_s",
        "elapsed_seconds", "elapsed_time_s"
    ],
    "deadline_missed": [
        "deadline_missed", "missed_deadline", "deadline_miss",
        "deadline_missed_flag", "missed_deadline_flag"
    ],
    "cycle_id": ["cycle_id", "cycle", "iteration", "iteration_id"],
    "run_id": ["run_id", "experiment_id", "trial_id"],
    "records": ["records_processed", "record_count", "num_records", "rows_processed"],
    "high_predictions": ["high_predictions", "num_high_predictions", "high_count"]
}

def find_column(key, frame=cycles):
    lookup = {norm_name(c): c for c in frame.columns}
    for alias in ALIASES.get(key, []):
        if norm_name(alias) in lookup:
            return lookup[norm_name(alias)]
    return None

def timing_to_ms(series, column_name):
    name = norm_name(column_name)
    values = pd.to_numeric(series, errors="coerce")
    if name.endswith("_ns") or "_ns_" in name:
        return values / 1_000_000
    if name.endswith("_us") or "_us_" in name:
        return values / 1_000
    if name.endswith("_s") and not name.endswith("_ms"):
        return values * 1_000
    return values

latency_col = find_column("total_latency")
interval_col = find_column("actual_interval")
configured_col = find_column("configured_interval")
timestamp_col = find_column("timestamp")
deadline_col = find_column("deadline_missed")
cycle_id_col = find_column("cycle_id")
run_id_col = find_column("run_id")

print("Detected columns:")
for label, value in [
    ("Total diagnosis/cycle latency", latency_col),
    ("Actual start-to-start interval", interval_col),
    ("Configured interval", configured_col),
    ("Timestamp / elapsed time", timestamp_col),
    ("Deadline-missed flag", deadline_col),
    ("Cycle ID", cycle_id_col),
    ("Run ID", run_id_col),
]:
    print(f"  {label}: {value}")

data = cycles.copy()

if latency_col:
    data["latency_ms"] = timing_to_ms(data[latency_col], latency_col)
if interval_col:
    data["actual_interval_ms"] = timing_to_ms(data[interval_col], interval_col)
if configured_col:
    data["configured_interval_ms"] = timing_to_ms(data[configured_col], configured_col)

if timestamp_col:
    raw_ts = data[timestamp_col]
    parsed_dt = pd.to_datetime(raw_ts, errors="coerce")
    if parsed_dt.notna().sum() >= max(1, int(0.5 * len(data))):
        data["plot_time"] = parsed_dt
        data = data.sort_values("plot_time")
    else:
        numeric_ts = pd.to_numeric(raw_ts, errors="coerce")
        data["plot_time"] = numeric_ts
        # Treat common elapsed-time fields as seconds; timestamps in ns are converted.
        tname = norm_name(timestamp_col)
        if tname.endswith("_ns") or "timestamp_ns" in tname:
            data["plot_time"] = numeric_ts / 1e9
        elif tname.endswith("_ms"):
            data["plot_time"] = numeric_ts / 1e3
        # Otherwise numeric values are assumed to be elapsed seconds or a monotonic clock.
else:
    data["plot_time"] = np.arange(len(data))

if cycle_id_col:
    data = data.sort_values(cycle_id_col, kind="stable")
else:
    data = data.reset_index(drop=True)

display(data.head())


## 2. Per-cycle latency and summary statistics

This is the primary diagnosis-latency analysis. Median and p95 are useful because occasional slow cycles can be hidden by the mean. The interpretation depends on what the benchmark runner times: ideally, total latency includes the complete diagnosis path, and stage timings are not added again if they are already included in the total.


In [ ]:
if "latency_ms" not in data.columns or data["latency_ms"].notna().sum() == 0:
    print("No recognized total-latency column found. Check the printed columns and update ALIASES.")
else:
    latency = data["latency_ms"].dropna()
    latency_stats = pd.Series({
        "Cycles with latency measurements": int(latency.count()),
        "Mean latency (ms)": latency.mean(),
        "Median latency (ms)": latency.median(),
        "P95 latency (ms)": latency.quantile(0.95),
        "P99 latency (ms)": latency.quantile(0.99),
        "Minimum latency (ms)": latency.min(),
        "Maximum latency (ms)": latency.max(),
        "Latency standard deviation (ms)": latency.std()
    })
    display(latency_stats.to_frame("Value").round(3))

    fig, ax = plt.subplots(figsize=(10, 4.8))
    x = data["plot_time"]
    ax.plot(x, data["latency_ms"], marker=".", linewidth=1)
    ax.axhline(latency.median(), linestyle="--", label=f"Median: {latency.median():.2f} ms")
    ax.axhline(latency.quantile(0.95), linestyle=":", label=f"P95: {latency.quantile(0.95):.2f} ms")
    ax.set_title("Diagnosis Latency Across Monitoring Cycles")
    ax.set_xlabel("Cycle timestamp / elapsed time")
    ax.set_ylabel("Total diagnosis latency (ms)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(latency, bins="auto", edgecolor="black")
    ax.axvline(latency.median(), linestyle="--", label=f"Median: {latency.median():.2f} ms")
    ax.axvline(latency.quantile(0.95), linestyle=":", label=f"P95: {latency.quantile(0.95):.2f} ms")
    ax.set_title("Distribution of Diagnosis Latency")
    ax.set_xlabel("Total diagnosis latency (ms)")
    ax.set_ylabel("Number of cycles")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


## 3. Actual monitoring interval versus configured interval

Use this plot to show whether the system actually produces diagnosis cycles at the intended cadence. The interval should be start-to-start time between consecutive cycles, not merely the duration of inference. If the benchmark runner does not record actual intervals, the notebook cannot infer them reliably from latency alone.


In [ ]:
if "actual_interval_ms" not in data.columns or data["actual_interval_ms"].notna().sum() == 0:
    print("No recognized actual monitoring-interval column found.")
else:
    observed = data["actual_interval_ms"].dropna()
    interval_stats = pd.Series({
        "Cycles with interval measurements": int(observed.count()),
        "Mean actual interval (ms)": observed.mean(),
        "Median actual interval (ms)": observed.median(),
        "P95 actual interval (ms)": observed.quantile(0.95),
        "Minimum actual interval (ms)": observed.min(),
        "Maximum actual interval (ms)": observed.max()
    })
    display(interval_stats.to_frame("Value").round(3))

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(data["plot_time"], data["actual_interval_ms"], marker=".", linewidth=1,
            label="Observed start-to-start interval")
    if "configured_interval_ms" in data.columns and data["configured_interval_ms"].notna().any():
        target = data["configured_interval_ms"].dropna().median()
        ax.axhline(target, linestyle="--", label=f"Configured interval (median): {target:.2f} ms")
    ax.set_title("Observed Monitoring Interval")
    ax.set_xlabel("Cycle timestamp / elapsed time")
    ax.set_ylabel("Start-to-start interval (ms)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    if "configured_interval_ms" in data.columns:
        paired = data[["actual_interval_ms", "configured_interval_ms"]].dropna()
        if not paired.empty:
            deviation = paired["actual_interval_ms"] - paired["configured_interval_ms"]
            print(f"Median interval deviation: {deviation.median():.2f} ms")
            print(f"P95 interval deviation: {deviation.quantile(0.95):.2f} ms")
            print("Note: a positive deviation means the observed interval exceeded the configured target.")


## 4. Deadline misses

Only use this metric if the benchmark runner defines a deadline clearly (for example, a cycle must finish within the configured monitoring interval). A missing flag is not treated as zero misses.


In [ ]:
if deadline_col is None:
    print("No deadline-missed flag recorded; deadline-miss rate is unavailable.")
else:
    raw = data[deadline_col]
    if raw.dtype == bool:
        missed = raw
    else:
        normalized = raw.astype(str).str.strip().str.lower()
        true_values = {"1", "true", "yes", "y", "missed", "deadline_missed"}
        false_values = {"0", "false", "no", "n", "not_missed", "on_time"}
        missed = normalized.map(lambda v: True if v in true_values else (False if v in false_values else np.nan))
    valid = missed.dropna()
    if len(valid) == 0:
        print("Deadline flag exists, but its values could not be interpreted. Inspect:", data[deadline_col].unique()[:20])
    else:
        rate = valid.astype(bool).mean() * 100
        print(f"Cycles with a valid deadline flag: {len(valid)}")
        print(f"Missed-deadline rate: {rate:.2f}%")
        print(f"Missed deadlines: {int(valid.astype(bool).sum())} / {len(valid)}")


## 5. Stage timing breakdown

This chart is optional. It is useful if the benchmark runner separately records map reading, feature engineering, Random Forest inference, MLP attribution, and output/reporting time. Stage timings should be measured independently; avoid double-counting nested timers.


In [ ]:
stage_groups = {
    "Map reading": ["map_read_ms", "map_read_ns", "map_reading_ms", "map_reading_ns"],
    "Aggregation": ["aggregation_ms", "aggregation_ns", "aggregate_ms", "aggregate_ns"],
    "Feature engineering": ["feature_engineering_ms", "feature_engineering_ns", "feature_processing_ms", "feature_processing_ns"],
    "Random Forest": ["rf_inference_ms", "rf_inference_ns", "random_forest_ms", "random_forest_ns", "rf_ms", "rf_ns"],
    "MLP attribution": ["mlp_attribution_ms", "mlp_attribution_ns", "attribution_ms", "attribution_ns"],
    "Output": ["output_ms", "output_ns", "reporting_ms", "reporting_ns"]
}

stage_rows = []
for stage, aliases in stage_groups.items():
    found = None
    lookup = {norm_name(c): c for c in data.columns}
    for alias in aliases:
        if norm_name(alias) in lookup:
            found = lookup[norm_name(alias)]
            break
    if found is not None:
        vals = timing_to_ms(data[found], found).dropna()
        if len(vals):
            stage_rows.append({
                "Stage": stage,
                "Median (ms)": vals.median(),
                "P95 (ms)": vals.quantile(0.95),
                "Mean (ms)": vals.mean()
            })

if not stage_rows:
    print("No recognized stage timing columns found.")
else:
    stage_stats = pd.DataFrame(stage_rows)
    display(stage_stats.round(3))
    fig, ax = plt.subplots(figsize=(9, 4.8))
    positions = np.arange(len(stage_stats))
    ax.bar(positions, stage_stats["Median (ms)"], yerr=None)
    ax.set_xticks(positions)
    ax.set_xticklabels(stage_stats["Stage"], rotation=25, ha="right")
    ax.set_ylabel("Median stage time (ms)")
    ax.set_title("Median Latency by Diagnosis Stage")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


## 6. CPU and memory overhead (if resource data was collected)

To quantify monitoring overhead, the resource CSV should distinguish baseline workload-only samples from monitored samples (for example, a `mode`, `condition`, or `monitoring_enabled` column). If both conditions are not present, the notebook can show the time series but cannot calculate a defensible overhead comparison.


In [ ]:
if resources is None:
    print("No system_resource_metrics.csv found; resource overhead cannot be analyzed.")
else:
    print("Resource columns:", resources.columns.tolist())
    r = resources.copy()
    rlookup = {norm_name(c): c for c in r.columns}

    def resource_col(candidates):
        for candidate in candidates:
            if norm_name(candidate) in rlookup:
                return rlookup[norm_name(candidate)]
        return None

    cpu_col = resource_col(["cpu_percent", "cpu_pct", "process_cpu_percent", "system_cpu_percent", "cpu_usage_percent"])
    mem_col = resource_col(["memory_mb", "rss_mb", "process_memory_mb", "memory_usage_mb", "rss_bytes", "memory_bytes"])
    mode_col = resource_col(["mode", "condition", "scenario", "monitoring_enabled", "experiment_mode"])
    rtime_col = resource_col(["elapsed_s", "elapsed_seconds", "timestamp", "elapsed_time_s"])

    print("Detected CPU:", cpu_col, "| memory:", mem_col, "| condition:", mode_col, "| time:", rtime_col)

    if mem_col:
        r["memory_plot_value"] = pd.to_numeric(r[mem_col], errors="coerce")
        if norm_name(mem_col).endswith("_bytes"):
            r["memory_plot_value"] = r["memory_plot_value"] / (1024 ** 2)
            memory_label = "Memory (MiB)"
        else:
            memory_label = "Memory value (column units)"
    if cpu_col:
        r["cpu_plot_value"] = pd.to_numeric(r[cpu_col], errors="coerce")

    if mode_col:
        print("\nSamples per condition:")
        display(r[mode_col].value_counts(dropna=False).to_frame("Samples"))

    if rtime_col:
        r["resource_plot_time"] = pd.to_numeric(r[rtime_col], errors="coerce")
    else:
        r["resource_plot_time"] = np.arange(len(r))

    if cpu_col:
        fig, ax = plt.subplots(figsize=(10, 4.5))
        if mode_col:
            for mode, grp in r.groupby(mode_col, dropna=False):
                ax.plot(grp["resource_plot_time"], grp["cpu_plot_value"], label=str(mode), linewidth=1)
            ax.legend()
        else:
            ax.plot(r["resource_plot_time"], r["cpu_plot_value"], linewidth=1)
        ax.set_title("CPU Usage During Benchmark")
        ax.set_xlabel("Elapsed time / sample index")
        ax.set_ylabel("CPU usage (%) as reported by collector")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

    if mem_col:
        fig, ax = plt.subplots(figsize=(10, 4.5))
        if mode_col:
            for mode, grp in r.groupby(mode_col, dropna=False):
                ax.plot(grp["resource_plot_time"], grp["memory_plot_value"], label=str(mode), linewidth=1)
            ax.legend()
        else:
            ax.plot(r["resource_plot_time"], r["memory_plot_value"], linewidth=1)
        ax.set_title("Memory Usage During Benchmark")
        ax.set_xlabel("Elapsed time / sample index")
        ax.set_ylabel(memory_label)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

    if mode_col and (cpu_col or mem_col):
        metric_map = {}
        if cpu_col:
            metric_map["CPU (%)"] = "cpu_plot_value"
        if mem_col:
            metric_map[memory_label] = "memory_plot_value"
        for label, col in metric_map.items():
            print(f"\n{label} by condition:")
            display(r.groupby(mode_col)[col].agg(["count", "mean", "median", "max"]).round(3))


## 7. Export compact analysis tables

These summaries can help when writing the paper. Keep the raw benchmark CSVs alongside them so the results remain traceable.


In [ ]:
analysis_dir = RESULTS_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

if "latency_ms" in data.columns:
    latency = data["latency_ms"].dropna()
    if len(latency):
        pd.DataFrame([{
            "n_cycles": int(len(latency)),
            "mean_latency_ms": latency.mean(),
            "median_latency_ms": latency.median(),
            "p95_latency_ms": latency.quantile(0.95),
            "p99_latency_ms": latency.quantile(0.99),
            "min_latency_ms": latency.min(),
            "max_latency_ms": latency.max(),
            "std_latency_ms": latency.std()
        }]).to_csv(analysis_dir / "latency_summary.csv", index=False)

if "actual_interval_ms" in data.columns:
    intervals = data["actual_interval_ms"].dropna()
    if len(intervals):
        pd.DataFrame([{
            "n_intervals": int(len(intervals)),
            "mean_actual_interval_ms": intervals.mean(),
            "median_actual_interval_ms": intervals.median(),
            "p95_actual_interval_ms": intervals.quantile(0.95),
            "min_actual_interval_ms": intervals.min(),
            "max_actual_interval_ms": intervals.max()
        }]).to_csv(analysis_dir / "interval_summary.csv", index=False)

print(f"Analysis outputs saved under: {analysis_dir.resolve()}")
print("No benchmark values are generated by this notebook; all statistics come from the CSV inputs.")


## Suggested wording for the paper

Use this only as a structure, and replace the brackets with measured values:

> **Real-Time Performance Evaluation:** We evaluated the runtime behavior of the framework under a concurrent mixed `stress-ng` workload. Across [N] measured diagnosis cycles, the median and 95th-percentile end-to-end diagnosis latencies were [X] ms and [Y] ms, respectively. The observed median start-to-start monitoring interval was [Z] ms, compared with the configured interval of [T] ms. The measured monitoring overhead was [CPU result] and [memory result] relative to the workload-only baseline.

Do not claim real-time responsiveness from inference latency alone. Report the observed monitoring interval too, and state whether map reads were periodic snapshots or session-end reads. If only one run was conducted, explicitly describe it as a single-run evaluation.
